#### difference between row_number(), rank(), dense_rank() & percent_rank()?

##### row_number()

- The **row_number()** function in PySpark is a window function that assigns a **unique sequential number, starting from 1**, to each row within a defined window partition.
- It is commonly used for `ranking` data, selecting the **top N records** within a group.

**When to Use row_number()**
- `Remove duplicates`
- `Keep latest record`
- `Deduplicate based on composite keys`
- `Implement SCD logic`

|     Function      |        Description	                            |                    Tie Handling                   |
|-------------------|-------------------------------------------------|---------------------------------------------------|
| **row_number()**	| Assigns a **unique sequential number** to each row.	| **Tied** values receive **different consecutive numbers** (e.g., 1, 2, 3, 4). |
| **rank()**	      | Assigns the **same rank** to **tied** values but leaves **gaps** in the sequence.	| Tied values receive the same rank, and subsequent ranks are **skipped** (e.g., 1, 2, 2, 4). |
| **dense_rank()**	| Assigns the **same rank** to **tied** values but **does not leave gaps**.	| Tied values receive the same rank, and subsequent ranks are **consecutive** (e.g., 1, 2, 2, 3). |
| **Percent Rank**  | percent_rank() returns a normalized rank between **0 and 1** based on row position within the partition | rank column contains values in a percentile form i.e. in the **decimal format**. |

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, rank, dense_rank, percent_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
# Define the schema
schema = StructType([
    StructField("employee_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True)
])

# Prepare the sample data
simple_data = [
    ("Karan", "Sales", 3500),
    ("Mahesh", "Sales", 4600),
    ("Rohith", "Sales", 4100),
    ("Vinod", "Sales", None),
    ("Satish", "Sales", 4600),
    ("Mridula", "Finance", 3000),
    ("Jagadish", "Sales", 3000),
    ("Swaroop", "Finance", 3300),
    ("Ganesh", "Finance", 3900),
    ("Hima", "Finance", 3900),
    ("Hima", "Finance", 3900),    
    ("Hitesh", "Finance", None),
    ("Hitesh", "Finance", 4500),
    ("Jyoti", "Marketing", 3000),
    ("Kumar", "Marketing", 2000),
    ("Kannan", "Sales", 4100),
    ("Swapna", "Admin", 5680)]

# Create a DataFrame
df = spark.createDataFrame(simple_data, schema=schema)

# Show the DataFrame
df.display()

employee_name,department,salary
Karan,Sales,3500
Mahesh,Sales,4600
Rohith,Sales,4100
Vinod,Sales,null
Satish,Sales,4600
Mridula,Finance,3000
Jagadish,Sales,3000
Swaroop,Finance,3300
Ganesh,Finance,3900
Hima,Finance,3900


##### 1) Add Column with Row Number to DataFrame by Partition
- `ORDER BY` clause is `mandatory` in the `window specification` when you use `ranking` window functions such as **ROW_NUMBER, RANK, and DENSE_RANK & PERCENT_RANK**.
- `Without ORDER BY`, these functions cannot determine the order of rows within each partition and will result in an `error 1`.
- For `analytic and aggregate window functions`, `ORDER BY is optional` unless the function's `logic requires` it.

In [0]:
# Applying partitionBy() and orderBy()
window_spec = Window.partitionBy("department").orderBy("salary")

# Add a new columns "row_number", "rank" & "dense_rank" over the specified window
result_df = df \
    .withColumn("row_number", row_number().over(window_spec)) \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("dense_rank", dense_rank().over(window_spec)) \
    .withColumn("percent_rank", percent_rank().over(window_spec))

# Show the result
result_df.display()

employee_name,department,salary,row_number,rank,dense_rank,percent_rank
Swapna,Admin,5680,1,1,1,0.0
Hitesh,Finance,null,1,1,1,0.0
Mridula,Finance,3000,2,2,2,0.16666666666666666
Swaroop,Finance,3300,3,3,3,0.3333333333333333
Ganesh,Finance,3900,4,4,4,0.5
Hima,Finance,3900,5,4,4,0.5
Hima,Finance,3900,6,4,4,0.5
Hitesh,Finance,4500,7,7,5,1.0
Kumar,Marketing,2000,1,1,1,0.0
Jyoti,Marketing,3000,2,2,2,1.0


**percent_rank():**
- is a window function that calculates the **relative rank (percentile) of a row** within a window partition.
- `It returns a value between 0 and 1`.
- The **first row** in any `partition` always has a percent_rank of **0**.
- If a partition has **only one row**, the **rank is 0**.

**Mandatory ORDER BY:**
- You cannot use percent_rank() without specifying an orderBy in your window.

**Tie Handling:**
- Rows with the same value in the orderBy column receive the same percent_rank.

**Nulls:**
- percent_rank() typically treats NULL values as the lowest values (if sorted ASC) and assigns them a rank of 0, though they are usually excluded from the actual percentage calculation in some SQL dialects.

##### 2) Add Row Number to the DataFrame without Partition

In [0]:
# Applying orderBy() without partitionBy()
window_wpart = Window.orderBy(col("salary"))

# Add a new columns "row_number", "rank" & "dense_rank" over the specified window
result_df_wprt = df \
    .withColumn("row_number", row_number().over(window_wpart)) \
    .withColumn("rank", rank().over(window_wpart)) \
    .withColumn("dense_rank", dense_rank().over(window_wpart)) \
    .withColumn("percent_rank", percent_rank().over(window_wpart))

display(result_df_wprt)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


employee_name,department,salary,row_number,rank,dense_rank,percent_rank
Vinod,Sales,null,1,1,1,0.0
Hitesh,Finance,null,2,1,1,0.0
Kumar,Marketing,2000,3,3,2,0.125
Mridula,Finance,3000,4,4,3,0.1875
Jagadish,Sales,3000,5,4,3,0.1875
Jyoti,Marketing,3000,6,4,3,0.1875
Swaroop,Finance,3300,7,7,4,0.375
Karan,Sales,3500,8,8,5,0.4375
Ganesh,Finance,3900,9,9,6,0.5
Hima,Finance,3900,10,9,6,0.5


##### 3) Filter duplicate records based on primary key

In [0]:
# Create sample data
data = [
    ("PartyA", "A1", "X", 100, "2025-09-01 10:00:00"),
    ("PartyA", "A2", "Y", 200, "2025-09-01 11:00:00"),   # later timestamp
    ("PartyB", "B1", "M", 300, "2025-09-01 09:00:00"),
    ("PartyB", "B2", "N", 400, "2025-09-01 12:00:00"),   # later timestamp
    ("PartyC", "C1", "P", 500, "2025-09-01 08:00:00")
]

# Define schema for readability
columns = ["product", "col1", "col2", "col3", "timestamp"]

df_row = spark.createDataFrame(data, columns)
display(df_row)

product,col1,col2,col3,timestamp
PartyA,A1,X,100,2025-09-01 10:00:00
PartyA,A2,Y,200,2025-09-01 11:00:00
PartyB,B1,M,300,2025-09-01 09:00:00
PartyB,B2,N,400,2025-09-01 12:00:00
PartyC,C1,P,500,2025-09-01 08:00:00


In [0]:
# Define a window specification to partition data by 'product' and order by 'timestamp'
window_spec = Window.partitionBy("product").orderBy("timestamp")

party_cols = ['col1', 'col2', 'col3']

df_final = (
    df_row \
        .withColumn("row_num", row_number().over(window_spec))
        .withColumn("rank", rank().over(window_spec))
        .withColumn("dense_rank", dense_rank().over(window_spec))
        .withColumn("percent_rank", percent_rank().over(window_spec))
)

display(df_final)

df_final = (
    df_row.withColumn("row_num", row_number().over(window_spec))
          .filter("row_num = 1")
)

display(df_final)

df_final_drop = (
    df_row.withColumn("row_num", row_number().over(window_spec))
          .filter("row_num = 1")
        #   .select(*party_cols)
          .drop("row_num")
)

display(df_final_drop)

product,col1,col2,col3,timestamp,row_num,rank,dense_rank,percent_rank
PartyA,A1,X,100,2025-09-01 10:00:00,1,1,1,0.0
PartyA,A2,Y,200,2025-09-01 11:00:00,2,2,2,1.0
PartyB,B1,M,300,2025-09-01 09:00:00,1,1,1,0.0
PartyB,B2,N,400,2025-09-01 12:00:00,2,2,2,1.0
PartyC,C1,P,500,2025-09-01 08:00:00,1,1,1,0.0


product,col1,col2,col3,timestamp,row_num
PartyA,A1,X,100,2025-09-01 10:00:00,1
PartyB,B1,M,300,2025-09-01 09:00:00,1
PartyC,C1,P,500,2025-09-01 08:00:00,1


product,col1,col2,col3,timestamp
PartyA,A1,X,100,2025-09-01 10:00:00
PartyB,B1,M,300,2025-09-01 09:00:00
PartyC,C1,P,500,2025-09-01 08:00:00


In [0]:
# Define a window specification to partition data by 'product' and order by 'timestamp'
window_spec = Window.partitionBy("product").orderBy(col("timestamp").desc())

df_final_drop = (
    df_row.withColumn("row_num", row_number().over(window_spec))
          .filter("row_num = 1")
        #   .select(*party_cols)
          .drop("row_num")
)

display(df_final_drop)

product,col1,col2,col3,timestamp
PartyA,A2,Y,200,2025-09-01 11:00:00
PartyB,B2,N,400,2025-09-01 12:00:00
PartyC,C1,P,500,2025-09-01 08:00:00
